This is an example of generating DayMet h5 files to drive ATS 2D transect simulations.

- Input
    - `data-processed/{site_name}/m2_coords_{site_name}.mat`
    - [optional] DayMet raw data, script can also download
- Output
    - `data-processed/{watershed_name}/{watershed_name}_DayMet_2013_2023.h5`
        - -> `outputs['daymet_filename_watershed']`
    - `data-processed/{site_name}/{site_name}_DayMet_2013_2023.h5`
        - -> `outputs['daymet_filename_site']`
    - `data-processed/{site_name}/{site_name}_DayMet_typical10yr_2013_2023.h5`
        - -> `outputs['daymet_spinup_filename_site']`

**File History**

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell -- schema-v2 forcing timeline configuration
from config_utils import (load_config, phase_dates, phase_label, noleap_day_of_year,
                          phase_forcing_dir, full_timeline_forcing_dir)
config = load_config('config.json')
case = config['case']; watershed_name = case['watershed_name']; hucs = case['hucs']; site_name = case['site_name']; meshsize_nx = case['meshsize_nx']
spinup_dates = phase_dates(config, 'spinup'); prefire_dates = phase_dates(config, 'prefire_transient')
postfire_dates = phase_dates(config, 'postfire_transient') if 'postfire_transient' in config else []
spinup_label = phase_label(config, 'spinup'); prefire_label = phase_label(config, 'prefire_transient')
postfire_label = phase_label(config, 'postfire_transient') if postfire_dates else None
nyears_steadystate_spinup = config['spinup']['steady_state_years']; nyears_cyclic_spinup = config['spinup']['cyclic_years']
forcing_spinup_dir = phase_forcing_dir(config, 'spinup'); forcing_prefire_dir = phase_forcing_dir(config, 'prefire_transient')
forcing_postfire_dir = phase_forcing_dir(config, 'postfire_transient') if postfire_dates else None
forcing_full_dir = full_timeline_forcing_dir(config)
for _d in (forcing_spinup_dir, forcing_prefire_dir, forcing_postfire_dir, forcing_full_dir):
    if _d is not None: _d.mkdir(parents=True, exist_ok=True)


In [ ]:
outputs={}

In [ ]:
import logging
import sys,os
sys.path.append(os.path.join(os.environ['ATS_SRC_DIR'],'tools','meshing','meshing_ats'))
# import meshing_ats
import h5py

import geopandas as gpd
import numpy as np
import pandas as pd
pd.options.display.max_columns = None
pd.options.display.max_rows = 20

import rasterio
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import fiona
import scipy.ndimage
import cftime, datetime

import matplotlib
import matplotlib.colors as colors
from matplotlib import pyplot as plt

import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions


# ats_input_spec library, to be moved to amanzi_xml
import ats_input_spec
import ats_input_spec.public
import ats_input_spec.io

# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors
from amanzi_xml.common.parameter import Parameter
from amanzi_xml.common.parameter_list import ParameterList
from ats_input_spec.public import known_specs

from scipy.io import loadmat
import h5py as h5

In [ ]:
# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('./data/soil_structure/GLHYMPS/GLHYMPS.shp')
sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('./data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)
#sources

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs_daymet = watershed_workflow.crs.daymet_crs()
crs_latlon = watershed_workflow.crs.latlon_crs() # essentially epsg(4269)
# note: epsg(4269) i.e. NAD83 vs epsg(4326) i.e. WGS84
# - https://gis.stackexchange.com/questions/170839/is-re-projection-needed-from-srid-4326-wgs-84-to-srid-4269-nad-83

# alternative
#proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84  = "epsg:4326" # latlon
#crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
#crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# Prepare watershed shape and hillslope shape

In [ ]:
# load from watershed shp
#watershed_name = 'OakCreek' # name the domain, used in filenames, etc
fname_watershed_shp = f'../data-processed/{watershed_name}/{watershed_name}_bounds.shp'
watershed_shape = gpd.read_file(fname_watershed_shp)

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
#meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon and shape object
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
xsec_plg_dict_shply = watershed_workflow.utils.create_shply(xsec_plg_dict)

# convert to latlon crs, used in some plots
reproj_xsec_plg = watershed_workflow.warp.shape(xsec_plg_dict, crs_daymet, crs_latlon)
reproj_xsec_plg_shply = watershed_workflow.utils.create_shply(reproj_xsec_plg)

In [ ]:
hillslope_gdf

# Get DayMet data downloaded for the watershed

originally copied from `OakCreek_ATS_3D/notebooks/data/meterology/daymet` to `OakCreek_ATS_2D/notebooks/data/meteorology/daymet`

In [ ]:
generate_daymet=True

#start_year = 2013 #1980
#end_year = 2023
#nyears_cyclic_steadystate = 10

In [ ]:
def read_dataset_from_hdf5(filename, calendar='noleap'):
    """Read an HDF5 file back into a watershed_workflow Dataset format.
    
    This reverses the operation done by watershed_workflow.io.write_dataset_to_hdf5
    
    Note: write_dataset_to_hdf5 reverses y to increasing order and flips data,
    so we need to reverse both back to match the original format.
    """
    import affine
    
    with h5py.File(filename, 'r') as fid:
        # Read time, x, y
        times_sec = fid['time [s]'][:]
        x = fid['x [m]'][:]
        y_increasing = fid['y [m]'][:]  # y is stored in increasing order in HDF5
        
        # Reverse y back to decreasing order (as expected by raster convention)
        y = y_increasing[::-1]
        
        # Get origin date from attributes
        origin_date_str = fid.attrs.get('origin date', '2000-1-1')
        origin_parts = origin_date_str.split('-')
        time0 = cftime.datetime(int(origin_parts[0]), int(origin_parts[1]), 
                               int(origin_parts[2]), calendar=calendar)
        
        # Convert times back to datetime objects (as numpy array)
        times = np.array([time0 + datetime.timedelta(seconds=int(t)) for t in times_sec])
        
        # Create profile (similar to rasterio profile)
        # y is now in decreasing order, so dy should be negative
        dx = x[1] - x[0] if len(x) > 1 else 1.0
        dy = y[0] - y[1] if len(y) > 1 else 1.0  # Should be positive since y is decreasing
        transform = affine.Affine(dx, 0, x[0], 0, -dy, y[0])  # dy is negative in transform
        
        profile = {
            'width': len(x),
            'height': len(y),
            'transform': transform,
            'crs': crs_daymet
        }
        
        # Create the dataset
        dataset = watershed_workflow.datasets.Dataset(profile, times)
        
        # Read each variable group
        for key in fid.keys():
            if key not in ['time [s]', 'x [m]', 'y [m]']:
                grp = fid[key]
                ntimes = len(times)
                data = np.zeros((ntimes, len(y), len(x)))
                
                for i in range(ntimes):
                    # Data was flipped when writing, so flip it back
                    # This matches with reversing y above
                    data[i, :, :] = np.flip(grp[str(i)][:], axis=0)
                
                dataset.data[key] = data
        
    return dataset


# --- Main logic: Load existing HDF5 or download new data ---
outputs['daymet_filename_watershed'] = '/global/cfs/cdirs/m1800/xiaoyi/10-Projects/2025-RCSFA-HillslopeFire/MaterialsData/Naches_from_zhi/Naches_DayMet_1980_2023.h5'

startdate_spinup = spinup_dates[0].isoformat()
enddate_spinup = (spinup_dates[-1] + datetime.timedelta(days=1)).isoformat()
startdate_transient = prefire_dates[0].isoformat()
enddate_transient = (prefire_dates[-1] + datetime.timedelta(days=1)).isoformat() # exclusive endpoint for watershed-workflow
bounds = tuple(watershed_shape.total_bounds) # if gpd format read from shp file

# Check if HDF5 file already exists
if os.path.exists(outputs['daymet_filename_watershed']):
    print(f"Loading existing DayMet data from: {outputs['daymet_filename_watershed']}")
    met_data_ats = read_dataset_from_hdf5(outputs['daymet_filename_watershed'])
    print(f"Loaded data with {len(met_data_ats.times)} time steps")
    print(f"Variables: {list(met_data_ats.data.keys())}")
    
elif generate_daymet:
    print("Downloading DayMet data (this may not work if download is unavailable)...")
    
    source = watershed_workflow.sources.manager_daymet.FileManagerDaymet()
    met_data = source.get_data(bounds, crs_daymet, startdate_spinup, enddate_transient)

    # watershed-workflow v1.5
    # unit conversion is included here, for prcp, mm/day -> m/s
    met_data_ats = watershed_workflow.daymet.convertToATS(met_data)
    attrs = watershed_workflow.daymet.getAttributes(bounds, startdate_spinup, enddate_transient)
    watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_filename_watershed'], met_data_ats, attrs)
    print(f"Downloaded and saved data to: {outputs['daymet_filename_watershed']}")
else:
    print("generate_daymet is False and file doesn't exist. Please set generate_daymet=True or provide the HDF5 file.")


In [ ]:
def convert_from_ATS_to_raw(met_data_ats):
    """Convert ATS format back to raw DayMet format for processing.
    
    This reverses the convertToATS operation to reconstruct the raw variables needed
    for interpolation and further processing.
    """
    profile = met_data_ats.profile
    times = met_data_ats.times
    met_data_raw = watershed_workflow.datasets.Dataset(profile, times)
    
    # Reverse the conversions done in convertToATS
    # Access data directly from the .data dictionary
    # air temperature: K -> C
    mean_air_temp_c = met_data_ats.data['air temperature [K]'] - 273.15
    met_data_raw.data['tmin'] = mean_air_temp_c  # approximation - we don't have separate min/max
    met_data_raw.data['tmax'] = mean_air_temp_c  # approximation
    
    # precipitation: m/s -> mm/day (combining rain and snow)
    precip_rain_ms = met_data_ats.data['precipitation rain [m s^-1]']
    precip_snow_ms = met_data_ats.data['precipitation snow [m SWE s^-1]']
    met_data_raw.data['prcp'] = (precip_rain_ms + precip_snow_ms) * 86400 * 1000  # m/s -> mm/day
    
    # shortwave radiation: need to reverse the dayl division
    # srad_Wm2 * dayl_s / 86400 = ATS value, so: srad_Wm2 = ATS value * 86400 / dayl_s
    # We don't have dayl, so we'll just use the ATS value directly
    met_data_raw.data['srad'] = met_data_ats.data['incoming shortwave radiation [W m^-2]']
    
    # vapor pressure - already in Pa
    met_data_raw.data['vp'] = met_data_ats.data['vapor pressure air [Pa]']
    
    # dayl - we don't have this, so create a placeholder (12 hours = 43200 seconds)
    met_data_raw.data['dayl'] = np.ones_like(mean_air_temp_c) * 43200
    
    return met_data_raw


# reprojected between crs. though dem and daymet actually have the same crs here
# code kept just in case

# If we loaded from HDF5, we need to convert back to raw format first
if 'met_data' not in locals():
    # We loaded from HDF5, so convert back
    met_data = convert_from_ATS_to_raw(met_data_ats)
    print("Converted ATS format back to raw DayMet format for processing")

met_data_warped = watershed_workflow.warp.dataset(met_data, dst_crs=crs_daymet)

**Modified Workflow:**
- If the HDF5 file already exists, load it directly (bypassing download)
- If not, attempt to download (original behavior)
- The loaded `met_data_ats` will be used for subsequent 2D transect processing

Unit of Daymet vars - https://daymet.ornl.gov/overview

| Parameter	| Abbr	| Units	| Description |
| ---- | ---- | ---- | ---- |
|Day length	|dayl	|s/day	|Duration of the daylight period in seconds per day. This calculation is based on the period of the day during which the sun is above a hypothetical flat horizon|
|Precipitation	|prcp	|mm/day	|Daily total precipitation in millimeters per day, sum of all forms converted to water-equivalent. Precipitation occurrence on any given day may be ascertained.|
|Shortwave radiation	|srad	|W/m2	|Incident shortwave radiation flux density in watts per square meter, taken as an average over the daylight period of the day. NOTE: Daily total radiation (MJ/m2/day) can be calculated as follows: ((srad (W/m2) * dayl (s/day)) / l,000,000)|
|Snow water equivalent	|swe	|kg/m2	|Snow water equivalent in kilograms per square meter. The amount of water contained within the snowpack.|
|Maximum air temperature	|tmax	|degrees C	|Daily maximum 2-meter air temperature in degrees Celsius.|
|Minimum air temperature	|tmin	|degrees C	|Daily minimum 2-meter air temperature in degrees Celsius.|
|Water vapor pressure	|vp	|Pa	|Water vapor pressure in pascals. Daily average partial pressure of water vapor.|

## Plot DayMet with 2D transect

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
loaded_data  = loadmat(m2_mat_filename)
start_coords = loaded_data['start_coords'].flatten()
end_coords   = loaded_data['end_coords'].flatten()

print(start_coords)
print(end_coords)

In [ ]:
# control xlim and ylim of plot
dx = 5000/2
dy = 4000/2
xmin = (start_coords[0]+end_coords[0])/2 - dx/2
xmax = (start_coords[0]+end_coords[0])/2 + dx/2
ymin = (start_coords[1]+end_coords[1])/2 - dy/2
ymax = (start_coords[1]+end_coords[1])/2 + dy/2

print([xmin, xmax, ymin, ymax])

In [ ]:
# Plot the datasets
# variables = ['tmin', 'tmax', 'prcp', 'srad', 'vp', 'dayl']
ivar = 'tmax'
islice = 100

fig = plt.figure()
ax1 = watershed_workflow.plot.get_ax(crs_daymet, fig, 1, 2, 1)
ax2 = watershed_workflow.plot.get_ax(crs_daymet, fig, 1, 2, 2)

if 'nodata' not in met_data_warped.profile:
    met_data_warped.profile['nodata'] = -9999

watershed_workflow.plot.raster(met_data_warped[ivar].profile, met_data_warped[ivar].data[islice,:,:], ax1)
watershed_workflow.plot.shply(xsec_plg_dict_shply, crs_daymet, ax=ax1, color='r')
#watershed_workflow.plot.hucs(watershed, crs_daymet, ax=ax1, color='r') # if watershed is from HUC
#watershed_workflow.plot.shplys(watershed_shape.geometry.iloc[0], crs_daymet, ax=ax1, color='r')
watershed_workflow.plot.shplys(watershed_shape.geometry, crs_daymet, ax=ax1, color='r')
ax1.set_title("Raw Daymet")

watershed_workflow.plot.raster(met_data_warped[ivar].profile, met_data_warped[ivar].data[islice,:,:], ax2)
watershed_workflow.plot.shply(xsec_plg_dict_shply, crs_daymet, ax=ax2, color='r')
ax2.set_title("Zoom in")
ax2.set_xlim(xmin, xmax)
ax2.set_ylim(ymin, ymax)

# Get Daymet for the hillslope site

modified from Bing's "2a-get_daymet.ipynb". First, extract 2D data. Then, generate typical year data for spinup.

## Find value at x and y in 2D transect

Use bilinear interpolation to get Daymet forcing at each location of the 2D transect

In [ ]:
from scipy import interpolate
from tqdm import tqdm
def bilinear_interpolate(x1,y1,x2,y2,dat1):
    dat2 = {}
    coord1 = np.concatenate([[x1.flatten()], [y1.flatten()]]).T
    coord2 = np.concatenate([[x2.flatten()], [y2.flatten()]]).T
    for varn, d1 in dat1.items():
        d1[d1==-9999] = np.nan 
        ntime, _, _ = d1.shape
        d2 = np.zeros([ntime, 2, len(x2)])
        for i in tqdm(range(ntime)):
            d2return = interpolate.griddata(coord1, d1[i,:,:].flatten(), coord2, method='linear')
            d2[i,0,:] = d2return
            d2[i,1,:] = d2return
            
        dat2[varn] = d2
    
        # Check nan values
        print("# of nan in interpolated {}: {}".format(varn, np.isnan(d2).sum()))
        
    return dat2

In [ ]:
# FASTER: crop met_data_warped.data before bilinear_interpolate

# Step 1: Get bounding box of gdf points
min_lon, max_lon = hillslope_gdf.lon.min(), hillslope_gdf.lon.max()
min_lat, max_lat = hillslope_gdf.lat.min(), hillslope_gdf.lat.max()

# Step 2: Convert bounding box to raster row/col indices
min_row, min_col = rasterio.transform.rowcol(met_data_warped.profile['transform'], min_lon, max_lat)
max_row, max_col = rasterio.transform.rowcol(met_data_warped.profile['transform'], max_lon, min_lat)

# Step 3: Expand by 3 rows/cols for interpolation
min_row = max(min_row - 3, 0)
max_row = min(max_row + 3, met_data_warped.data['tmin'].shape[1] - 1)  # Ensure within bounds
min_col = max(min_col - 3, 0)
max_col = min(max_col + 3, met_data_warped.data['tmin'].shape[2] - 1)  # Ensure within bounds

# Step 4: Extract the relevant subset of data
subset_data = {
    var: met_data_warped.data[var][:, min_row:max_row+1, min_col:max_col+1]
    for var in met_data_warped.data.keys()
}

# Step 5: Generate row/col indices for the subset
first_var = next(iter(subset_data))  # Get the first key
rows, cols = subset_data[first_var].shape[1:]  # Assuming it's a 3D array (time, row, col)
row_indices, col_indices = np.meshgrid(np.arange(min_row, max_row+1), np.arange(min_col, max_col+1), indexing="ij")

# Step 6: Convert row/col indices to x, y coordinates
xs, ys = rasterio.transform.xy(met_data_warped.profile['transform'], row_indices, col_indices, offset="center")
xs = np.array(xs)
ys = np.array(ys)

# Step 7: Flatten for interpolation
xmesh_flatten = xs.flatten()
ymesh_flatten = ys.flatten()
raw_dat_2dtran = bilinear_interpolate(xmesh_flatten, 
                                      ymesh_flatten, 
                                      hillslope_gdf.lon.values, 
                                      hillslope_gdf.lat.values, 
                                      subset_data)

In [ ]:
raw_dat_2dtran['prcp'].shape # dim=(time, y, x)

In [ ]:
# Plot spatial averaged rainfall
for varn in raw_dat_2dtran:
    fig,ax = plt.subplots(1,1,figsize=(15,3))
    ax.plot(raw_dat_2dtran[varn].mean(axis=(1,2)), '*-')
    ax.set(ylabel=varn)

In [ ]:
# noticing, raw_dat_2dtran is interpolated from met_data_warped, so prcp unit is mm/d

print("Maximum value: " + str(max(raw_dat_2dtran['prcp'].mean(axis=(1,2)))))
print("Minimum value: " + str(min(raw_dat_2dtran['prcp'].mean(axis=(1,2)))))
print("Sum of values: " + str(sum(raw_dat_2dtran['prcp'].mean(axis=(1,2)))))
print("Average value: " + str(sum(raw_dat_2dtran['prcp'].mean(axis=(1,2)))/raw_dat_2dtran['prcp'].shape[0]))

In [ ]:
varn='prcp'
fig,ax = plt.subplots(1,1,figsize=(8,3))

times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in met_data_ats.times])

ax.plot(times, raw_dat_2dtran[varn].mean(axis=(1,2)), '*-')
ax.set(ylabel='prcp [mm/d]')

## Generate the typical year DayMet for spinup

- note for spinup DayMet
    - in ww v1.5 Coweeta example, there are two approaches, average and median
    - in Bing's notebook, it just take the 1st year repeatly
    - in Zhi's notebook, it uses the averaged
- note for DayMet x and y
    - 

In [ ]:
# target x and y used in DayMet h5 file for 2D transect ATS
x_2dtran, y_2dtran = hillslope_gdf.h_distance.values.flatten(), np.array([0.0, 1.0])
print(x_2dtran)
print(y_2dtran)

In [ ]:
import copy
raw_dat_2dtran_warped = copy.deepcopy(met_data_warped)

# update dataset.data
raw_dat_2dtran_warped.data = raw_dat_2dtran
# update dataset.profile
raw_dat_2dtran_warped.profile['width'] = len(x_2dtran)
raw_dat_2dtran_warped.profile['height'] = len(y_2dtran)

from affine import Affine
dx_2dtran = (x_2dtran[-1] - x_2dtran[0])/meshsize_nx #x_2dtran[1] - x_2dtran[0]
new_transform = Affine(dx_2dtran, 0, 0,  # (a, b, c)
                       0, -1, len(y_2dtran)-1)    # (d, e, f)
raw_dat_2dtran_warped.profile['transform'] = new_transform

## verify the Affine here, related to watershed_workflow.io.write_dataset_to_hdf5 below
# print(raw_dat_2dtran_warped.data['prcp'].shape) # -> num_timeslice, num_row, num_col, e.g. (4015, 2, 101)
## the target x = 0:dx:680, y=[0,1]
## Affine defines the transformation from [col, row] to [x,y]
## x = a*col + b*row + c
## y = d*col + e*row + f
x = np.array([(new_transform * (i, 0))[0] for i in range(len(x_2dtran))])
y = np.array([(new_transform * (0, j))[1] for j in range(len(y_2dtran))])
print(x)
print(y) # [note] y will be reverted in watershed_workflow.io.write_dataset_to_hdf5

In [ ]:
bounds = [x_2dtran[0], y_2dtran[0], x_2dtran[-1], y_2dtran[-1]]
print(bounds)

In [ ]:
def _idx(day):
    stamp = cftime.DatetimeNoLeap(day.year, day.month, day.day, has_year_zero=True)
    found = np.where(raw_dat_2dtran_warped.times == stamp)[0]
    if len(found) != 1: raise ValueError(f'DayMet does not contain {day.isoformat()}')
    return found[0]
index_startdate_spinup, index_enddate_spinup = _idx(spinup_dates[0]), _idx(spinup_dates[-1])
index_startdate_transient, index_enddate_transient = _idx(prefire_dates[0]), _idx(prefire_dates[-1])
if postfire_dates: index_startdate_postfire, index_enddate_postfire = _idx(postfire_dates[0]), _idx(postfire_dates[-1])


In [ ]:
def _slice(start, end):
    d = watershed_workflow.datasets.Dataset(raw_dat_2dtran_warped.profile, raw_dat_2dtran_warped.times[start:end + 1])
    for key in raw_dat_2dtran_warped.data: d.data[key] = raw_dat_2dtran_warped.data[key][start:end + 1, :, :]
    return d
raw_dat_2dtran_warped_spinup = _slice(index_startdate_spinup, index_enddate_spinup)
raw_dat_2dtran_warped_transient = _slice(index_startdate_transient, index_enddate_transient)
raw_dat_2dtran_warped_postfire = _slice(index_startdate_postfire, index_enddate_postfire) if postfire_dates else None


## Write to ATS/HDF5 format for transient run

In [ ]:
raw_dat_2dtran_warped_transient_ats = watershed_workflow.daymet.convertToATS(raw_dat_2dtran_warped_transient)
outputs['daymet_prefire_filename_site'] = str(forcing_prefire_dir / f'{site_name}_DayMet.h5')
watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_prefire_filename_site'], raw_dat_2dtran_warped_transient_ats, watershed_workflow.daymet.getAttributes(bounds, prefire_dates[0].isoformat(), prefire_dates[-1].isoformat()))
raw_dat_2dtran_warped_postfire_ats = None
if postfire_dates:
    raw_dat_2dtran_warped_postfire_ats = watershed_workflow.daymet.convertToATS(raw_dat_2dtran_warped_postfire)
    outputs['daymet_postfire_filename_site'] = str(forcing_postfire_dir / f'{site_name}_DayMet.h5')
    watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_postfire_filename_site'], raw_dat_2dtran_warped_postfire_ats, watershed_workflow.daymet.getAttributes(bounds, postfire_dates[0].isoformat(), postfire_dates[-1].isoformat()))


In [ ]:
# print(met_data_ats.times)
# print(met_data_warped.times)
# print(raw_dat_2dtran_warped.times)
# print(raw_dat_2dtran_warped_ats.times)

In [ ]:
# plot the transient DayMet h5 file for ATS
fig = plt.figure(figsize=(15,6))
ax = fig.add_subplot(221)

times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in raw_dat_2dtran_warped_transient_ats.times])
prain_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['precipitation rain [m s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
psnow_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['precipitation snow [m SWE s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, prain_spatial_mean_raw, 'b', label='rain')
ax.plot(times, psnow_spatial_mean_raw, 'c', label='snow')
ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(222)
qswin_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['incoming shortwave radiation [W m^-2]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, qswin_spatial_mean_raw, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')

ax = fig.add_subplot(223)
airtemp_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['air temperature [K]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, airtemp_spatial_mean_raw, 'g')
ax.set_ylabel('air temperature [K]')

ax = fig.add_subplot(224)
vapor_pressure_spatial_mean_raw = raw_dat_2dtran_warped_transient_ats['vapor pressure air [Pa]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, vapor_pressure_spatial_mean_raw, 'm')
ax.set_ylabel('vapor pressure air [Pa]')
plt.show()

## Write to ATS/HDF5 format for cyclic spinup

This will write daymet in a format that ATS can read. E.g., this will partition precipitation into rain and snow, convert vapor pressure to relative humidity, get mean air temperature and so on.

- dout has dims of `(ntime, nrow, ncol)` or `(ntime, ny, nx)`, where nrow = ny = 1

### Water-year interpretation

The raw DayMet records used to create this typical year span the configured spinup source period, currently **2012-10-01 through 2017-09-30**.  This is five complete October-to-September water years on the no-leap calendar. `computeAverageYear()` creates one 365-day representative water-year cycle and repeats it for the configured cyclic-spinup duration (currently 10 years).

The returned cftime labels may begin at `2000-01-01`. That is a synthetic typical-year origin: its first day represents the first day of the selected water-year cycle (October 1), not a claim that the raw DayMet forcing began on January 1. ATS uses the ordered forcing values and, in the merged HDF5, the zero-based `time [s]` axis.


In [ ]:
# compute the typical year of the _raw_ data
# note that we set interpolate to False, since met_data is already daily on a noleap calendar
dat_2dtran_spinup_smooth = watershed_workflow.datasets.computeAverageYear(raw_dat_2dtran_warped_spinup,
                                                                          nyears_cyclic_spinup,
                                                                          smooth=True, 
                                                                          smooth_kwargs=dict(window_length=181, polyorder=2),
                                                                          interpolate=False)
# convert that to ATS
dat_2dtran_spinup_smooth_ats = watershed_workflow.daymet.convertToATS(dat_2dtran_spinup_smooth)

In [ ]:
# Diagnostic only: computeAverageYear uses a synthetic 2000-01-01 origin.
# Its first day represents the configured water-year cycle's Oct. 1 start;
# it does not change the raw 2012-10-01..2017-09-30 DayMet source period.
print(dat_2dtran_spinup_smooth_ats.times)


In [ ]:
# plot the smoothed precip result
fig = plt.figure(figsize=(15,3))
ax = fig.add_subplot(121)

times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in dat_2dtran_spinup_smooth_ats.times])
prain_spatial_mean_smooth = dat_2dtran_spinup_smooth_ats['precipitation rain [m s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
psnow_spatial_mean_smooth = dat_2dtran_spinup_smooth_ats['precipitation snow [m SWE s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, prain_spatial_mean_smooth, 'b', label='rain')
ax.plot(times, psnow_spatial_mean_smooth, 'c', label='snow')

ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(122)
qswin_spatial_mean_smooth = dat_2dtran_spinup_smooth_ats['incoming shortwave radiation [W m^-2]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, qswin_spatial_mean_smooth, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')
plt.show()

In [ ]:
# from ww1.5 coweeta demo
# often smoothing precip like that is a bad idea -- you now have every day misting with low intensity rain which can result in a ton of interception
# and canopy evaporation and no transpiration.  Another approach is to just take the median total rainfall year and repeat that year multiple times.
precip_raw = raw_dat_2dtran_warped_spinup['prcp'].data
shape_xy = precip_raw.shape[1:]
precip_raw = precip_raw.reshape((-1, 365,)+shape_xy)
annual_precip_raw = precip_raw.sum(axis=(1,2,3))

# note -- don't use np.median here... for even number of years it will not appear.  Instead, sort and talk the halfway point
median_i = sorted(((i,v) for (i,v) in enumerate(annual_precip_raw)), key=lambda x : x[1])[len(annual_precip_raw)//2][0]
typical_precip_raw = np.tile(precip_raw[median_i], (nyears_cyclic_spinup,1,1))
dat_2dtran_spinup_smooth['prcp'] = typical_precip_raw

# convert that to ATS
dat_2dtran_spinup_smooth_ats = watershed_workflow.daymet.convertToATS(dat_2dtran_spinup_smooth)

In [ ]:
# plot again, different precip typical year
fig = plt.figure(figsize=(15,3))
ax = fig.add_subplot(121)

times = np.array([datetime.datetime(*t.timetuple()[0:6]) for t in dat_2dtran_spinup_smooth_ats.times])
prain_spatial_mean_spinup = dat_2dtran_spinup_smooth_ats['precipitation rain [m s^-1]'].data.mean(axis=(1,2))#.data[:,0,5]
psnow_spatial_mean_spinup = dat_2dtran_spinup_smooth_ats['precipitation snow [m SWE s^-1]'].data.mean(axis=(1,2))#data[:,0,5]
ax.plot(times, prain_spatial_mean_spinup, 'b', label='rain')
ax.plot(times, psnow_spatial_mean_spinup, 'c', label='snow')

ax.set_ylabel('precip [m s^-1]')
ax.legend()

ax = fig.add_subplot(122)
qswin_spatial_mean_spinup = dat_2dtran_spinup_smooth_ats['incoming shortwave radiation [W m^-2]'].data.mean(axis=(1,2))#.data[:,0,5]
ax.plot(times, qswin_spatial_mean_spinup, 'r')
ax.set_ylabel('incoming shortwave radiation [W m^-2]')
plt.show()

In [ ]:
outputs['daymet_spinup_filename_site'] = str(forcing_spinup_dir / f'{site_name}_DayMet_cyclic{nyears_cyclic_spinup}y.h5')
watershed_workflow.io.write_dataset_to_hdf5(outputs['daymet_spinup_filename_site'], dat_2dtran_spinup_smooth_ats, watershed_workflow.daymet.getAttributes(bounds, spinup_dates[0].isoformat(), spinup_dates[-1].isoformat()))


In [ ]:
# Calculate the mean values (equivalent to your original calculation)
prain_spatial_temporal_mean = np.mean(prain_spatial_mean_spinup)
psnow_spatial_temporal_mean = np.mean(psnow_spatial_mean_spinup)
mean_precip4ats = prain_spatial_temporal_mean + psnow_spatial_temporal_mean

print(f'Mean annual precip rate [m s^-1] = {mean_precip4ats}')

## Merged data for ats-pflotran transient
- mainly for restart use

In [ ]:
outputs['daymet_full_timeline_filename_site'] = str(forcing_full_dir / f'{site_name}_DayMet.h5')


In [ ]:
phases = [('cyclic spinup', dat_2dtran_spinup_smooth_ats, nyears_cyclic_spinup * 365), ('prefire transient', raw_dat_2dtran_warped_transient_ats, len(prefire_dates))]
if postfire_dates: phases.append(('postfire transient', raw_dat_2dtran_warped_postfire_ats, len(postfire_dates)))
for label, dataset, expected in phases:
    if len(dataset.times) != expected: raise ValueError(f'{label}: expected {expected}, found {len(dataset.times)}')
total = sum(len(dataset.times) for _, dataset, _ in phases); times = np.arange(total) * 86400
print('\nDayMet forcing summary'); print(f'  spinup source period: {spinup_label} ({len(spinup_dates)} days)'); print(f'  cyclic ATS spinup: {nyears_cyclic_spinup} years ({len(phases[0][1].times)} days)'); print(f'  prefire transient: {prefire_label} ({len(prefire_dates)} days)'); print(f'  postfire transient: {postfire_label} ({len(postfire_dates)} days)' if postfire_dates else '  postfire transient: not configured'); print(f'  full timeline: {total} days; time = {times[0]} to {times[-1]} s')
with h5.File(outputs['daymet_full_timeline_filename_site'], 'w') as hdf:
    hdf.create_dataset('time [s]', data=times); hdf.create_dataset('x [m]', data=x_2dtran); hdf.create_dataset('y [m]', data=y_2dtran)
    for key in phases[0][1].data:
        data = np.concatenate([dataset.data[key] for _, dataset, _ in phases], axis=0)
        if len(data) != total: raise ValueError(f'DayMet mismatch: {key}')
        group = hdf.create_group(key)
        for i, value in enumerate(data): group.create_dataset(str(i), data=value)
print(f'Wrote canonical DayMet forcing: {outputs["daymet_full_timeline_filename_site"]}')


In [ ]:
outputs